<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->
# Ground Station Visibility
---
Last revised by Z. Ellis on 2026 APR 6

## Objectives
This tutorial will demonstrate 

## See Also
- [Celestial Bodies](basics_CelestialBodies.ipynb)
- [Keplerian Orbit Propagation](basics_PropagatorKeplerian.ipynb)

## Imports and Set Up
Here we'll import the necessary libraries and load in the tutorials data folder. Then we define units and frames and load a metakernel for SPICE functionalities.

In [1]:
import scarabaeus as scb
import supplementary as supp

import os
import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = supp.load_data()

## units, frames, kernels
km, kg, sec, hr = scb.Units.get_units(['km', 'kg', 'sec', 'hr'])
J2000 = scb.Frame('J2000')

# load tutorial meta kernel
scb.SpiceManager.clear_kernels()    # ensure clean kernel pool
scb.SpiceManager.load_kernel_from_mkfile(data.mk.path)

SCB tutorial data up to date.


## Configure Bodies and Propagate Trajectory
Now we'll have to create an orbiter spacecraft as well as the Earth for it to orbit around.

First we create the Earth using Scarabaeus' built in constants and attach DSN stations to it using IDs from the [NAIF Integer ID Code list](https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/C/req/naif_ids.html#Ground%20Stations.):

In [2]:
## configure primary body
# create an empty Earth
earth = scb.CelestialBody.from_constants('EARTH')

# define a few ground stations and attach them to Earth
DSS_14 = scb.GroundStation('DSS-14', 399014)
DSS_15 = scb.GroundStation('DSS-15', 399015)
DSS_24 = scb.GroundStation('DSS-24', 399024)
DSS_34 = scb.GroundStation('DSS-34', 399034)
DSS_54 = scb.GroundStation('DSS-54', 399054)

all_gs = [DSS_14, DSS_15, DSS_24, DSS_34, DSS_54]
earth.add_ground_station(all_gs)

AttributeError: 'CelestialBody' object has no attribute 'add_ground_station'

Now we'll create a simple point mass spacecraft. Since we're only using Keplerian dynamics, we won't need to configure anything else for it:

In [3]:
## configure the orbiter spacecraft
# create Spacecraft object
orbiter = scb.Spacecraft(name     = 'ORCCA_ORBITER',
                         spice_id = -1000,
                         tot_mass = scb.ArrayWUnits(2e3, kg))

With both bodies created, we're ready to set up the propagation. First define the interval we'd like to examine, for this tutorial we'll look at a week's worth of passes and use a time step of 1 hour to keep the propagation quick. Then we'll create the initial state, placing the Orcca Orbiter in a circular LEO orbit.

In [4]:
# define propagation interval
t0 = scb.EpochArray(scb.SpiceManager.cal2et('2026 JAN 01 00:00:00.000'), 'TDB')
tf = scb.EpochArray(scb.SpiceManager.cal2et('2026 JAN 07 00:00:00.000'), 'TDB')
dt = scb.ArrayWUnits(1, hr)

epochs = scb.EpochArray.interval(t0, tf, dt)

# define initial state
R = scb.ArrayWUnits(8e3, km)
v_circ = scb.ArrayWUnits(np.sqrt((earth.grav_param.values) / R.values), km/sec)

pos0 = scb.ArrayWFrame(scb.ArrayWUnits([10e6, 0, 0], km    ), J2000)
vel0 = scb.ArrayWFrame(scb.ArrayWUnits([0   , 7, 0], km/sec), J2000)
x0   = scb.StateArray(epoch  = t0,
                      origin = earth,
                      state  = scb.StateDefinition()
                                  .position(orbiter, pos0)
                                  .velocity(orbiter, vel0))

TypeError: Equality operation requires both instances to be of type ArrayWUnits.

Finally, we'll propgate the orbiter with Keplerian dynamics and save its trajectory to an SPK, the file `ORCCA_ORBITER_TRAJ.bsp`, so that we can plot the visibility in the next section.

In [5]:
## propagate the orbiter
# define keplerian dynamics, propagator, and propagate
kep_dyn = scb.ForceModelTranslation(orbiter)
prop    = scb.Propagator(primary_body = orbiter,
                         state_vector = x0,
                         tspan        = epochs,
                         force_models = kep_dyn)
prop.propagate()

## write trajectory file
# create directory in tutorial results folder if it doesn't exist yet
gs_vis_tut_path = os.path.join(tutorial_data.RESULTS_PATH, 'GS_vis_tut/')
if not os.path.exists(gs_vis_tut_path): 
    os.mkdir(gs_vis_tut_path)

# remove file if it exists from previous run
# orbit_traj_path = os.path.join(gs_vis_tut_path, 'ORCCA_ORBITER_TRAJ.bsp')
# if os.path.exists(orbit_traj_path): os.remove(orbit_traj_path)

# # write trajectory
# orbit_traj = scb.Trajectory(orbit_traj_path, state_array = prop.propagated_state_array)

NameError: name 'x0' is not defined

## Examine Visibility
Let's look at the visibility windows using the propagated SPK we saved in the cell above. First, we need to get a window for each ground station by looping through the Earth's GS list across the entire time interval. Once that's done, we just need to pass it to the `plot_gs_visibility` plotting function to see when the Orcca Orbiter is visible for which ground station.

In [6]:
# compute visibility windows for each ground station
vis_windows = {}
for gs in earth.gs_list:
    # don't need number of windows here -> ignore with _
    _, windows = scb.SpiceManager.get_observer_target_visibility_windows(
                                            obsvr_bdy   = gs,
                                            trgt_bdy    = orbiter,
                                            epoch_start = epochs[0],
                                            epoch_end   = epochs[-1],
                                            step_size   = 60)
    # save windows
    vis_windows[gs] = windows

vis_fig = scb.Plotting.plot_gs_visibility(vis_windows, epochs[0].times.values)
plt.show()

NameError: name 'epochs' is not defined